# Zero-shot TAD / DADA-2000 — **frame archives** (CPU, run once)

Rule and design: `core/docs/final/ZS_TAD_DADA.md` §4. SA² needs pixels, and the frames of both sets are expensive to
reach: TAD only from Kaggle, DADA-2000 only inside the 117 GiB spanned `DADA2000.zip` on Drive (370 sources ≈ 18 GiB of
PNG). Only `frames[::8]` become rows, so this notebook keeps **only those**, once, on a **CPU runtime** — the GPU
notebook (`zs_tad_dada_r4.ipynb`) then unpacks ≈ 2.5 GB and never waits on the mount (the DoTA-CAP pattern).

| step | what | output |
|---|---|---|
| 1–3 | setup, deps, helpers, ids (TAD 100 test videos, DADA 370 T2-test sources) | — |
| 4 | **TAD**: Kaggle → numeric-order farm → `zs_frames pack` (`frames[::8]`, gate: = `TAD_ncc` rows) → `verify` | `Thesis-Final/data/ZS_frames/TAD/` |
| 5 | **DADA**: `7z l` listing once → `zs_frames dada-plan` (row frames only, census-checked) → `7z x @list` → `pack --prestrided` → `verify` | `Thesis-Final/data/ZS_frames/DADA2000_orig/` |

Each `{id}.zip` holds the row frames' **original bytes** (stored). `verify` writes `frames_manifest.json` only when every
video holds exactly its CLIP rows. Every step skips what is already archived — **re-run after a disconnect**.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data (TAD, DADA2000_orig, DADA2000Origin), cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, A2 bakes (Nexar stats), KNN, v2 A2 reads
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code + every archive / cache / output this notebook writes
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'     # new caches (clip_windows/, clip_patches/) land here
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'ZS_ARMS'), 'this upload predates the zero-shot TAD / DADA build -- re-upload kat-vad/'
assert (REPO / 'core/docs/final/ZS_TAD_DADA.md').is_file(), 'no ZS_TAD_DADA.md (the read-out rule) in this upload'

STRIDE   = constants.ZS_STRIDE                                            # 8
TAD      = constants.DATA_ROOT / constants.TAD_DATASET                    # Thesis/data/TAD (labels s8)
TAD_NCC  = Path(DRIVE) / 'cache' / 'clip' / 'TAD_ncc'                    # T0's CLIP rows = frames[::8]
T2NAME   = constants.DADA_ORIGIN_DATASET                                  # DADA2000_orig
T2       = constants.DATA_ROOT / T2NAME                                   # meta.json (T2-test windows)
T2_CLIP  = Path(DRIVE) / 'cache' / 'clip' / T2NAME                       # per-source CLIP rows at s8
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ORIG     = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME
ARCHIVE  = Path(DRIVE_F) / 'data' / constants.ZS_FRAMES_DIRNAME            # written by zs_pack_frames.ipynb
ARC      = {'tad': ARCHIVE / constants.TAD_DATASET, 'dada': ARCHIVE / T2NAME}
CLIP     = {'tad': TAD_NCC, 'dada': T2_CLIP}
ROOT     = constants.OUTPUT_ROOT / constants.ZS_OUTPUT_DIRNAME            # Thesis-Final/outputs/zs
REPORTS  = ROOT / 'REPORTS'

LOCAL    = Path('/content/zs_pack')
KAGGLE   = 'nikanvasei/traffic-anomaly-dataset-tad'                     # tad_zeroshot.ipynb's source
for p in (REPORTS, LOCAL, ARC['tad'], ARC['dada']):
    p.mkdir(parents=True, exist_ok=True)
for name, path in (('TAD labels', TAD / constants.FRAME_LABELS_TEST_FILENAME), ('TAD_ncc', TAD_NCC),
                   ('T2 meta', T2 / constants.META_FILENAME), ('T2 CLIP', T2_CLIP), ('T2 census', T2_CENSUS),
                   ('DADA2000.zip', ORIG / 'DADA2000.zip')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:13s} {path}')
print('archives ->', ARCHIVE)

In [ ]:
%%bash
apt-get -qq install -y p7zip-full > /dev/null
pip install -q "transformers==4.56.*" av einops faiss-cpu kagglehub openpyxl xlrd   # core imports them
7z | head -2
df -h /content | tail -1

In [ ]:
import pickle
import shutil
import subprocess
import time

import numpy as np

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF (absl eats INFO logs)


def run(cmd, log=None, quiet_steps=False, stdout_to=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    out = open(stdout_to, 'w') if stdout_to else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                if out:
                    out.write(line)
                    continue
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                if not (quiet_steps and 'epoch' in line and 'batch' in line):
                    print(line, end='')
    finally:
        if sink:
            sink.close()
        if out:
            out.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def flush_drive():
    t0 = time.time()
    drive.flush_and_unmount()
    drive.mount('/content/drive')
    print(f'Drive flushed and remounted in {time.time() - t0:.0f}s')


def rows(d, v):
    return int(np.load(d / f'{v}.npy', mmap_mode='r').shape[0])

from core.data.v2_splits import lines_sha1             # noqa: E402
from core.tools.dada_whole_eval import test_sources    # noqa: E402

labels_s8 = json.loads((TAD / constants.FRAME_LABELS_TEST_FILENAME).read_text())
IDS = {'tad': sorted(labels_s8), 'dada': test_sources(T2)}
n_abn = sum(int(max(g) > 0) for g in labels_s8.values())
assert (len(IDS['tad']), n_abn) == (constants.V2_TADZ_VIDEOS, constants.V2_TADZ_ABNORMAL), (len(IDS['tad']), n_abn)
assert len(IDS['dada']) == 370, len(IDS['dada'])
for s in ('tad', 'dada'):
    miss = [v for v in IDS[s] if not (CLIP[s] / f'{v}.npy').exists()]
    assert not miss, f'{s}: {len(miss)} videos have no CLIP rows in {CLIP[s]}: {miss[:5]}'
bad = [v for v in IDS['tad'] if rows(TAD_NCC, v) != len(labels_s8[v])]
assert not bad, f'TAD_ncc rows != s8 labels on {bad[:5]}'
ID_FILE = {s: write_ids(LOCAL / f'{s}_ids.txt', IDS[s]) for s in IDS}


def archive_ok(s):
    """The verify manifest exists and covers exactly this id list (zs_frames verify, PASS only)."""
    m = ARC[s] / constants.ZS_FRAMES_MANIFEST
    if not m.is_file():
        return False
    man = json.loads(m.read_text())
    return man['pass'] and man['ids_sha1'] == lines_sha1(IDS[s]) and man['stride'] == STRIDE


print(f"TAD {len(IDS['tad'])} test videos ({n_abn} abnormal, {sum(rows(TAD_NCC, v) for v in IDS['tad'])} rows) | "
      f"DADA {len(IDS['dada'])} T2-test sources ({sum(rows(T2_CLIP, v) for v in IDS['dada'])} rows)")
print({s: ('archived' if archive_ok(s) else 'NOT archived') for s in IDS})

## 4. TAD — Kaggle → farm → archive (≈ 11k row frames)

TAD's Kaggle frame names are not zero-padded (`10.jpg` sorts before `2.jpg`, TAD_SETUP §4.3): the farm links every frame
of the 100 test videos under `{numeric index:06d}{ext}`, exactly `tad_zeroshot.ipynb`'s farm (its T9(c) gate passed at
cos ≥ 0.999). `zs_frames pack` keeps `frames[::8]` and refuses a video whose count is not its `TAD_ncc` rows.

In [ ]:
import math
import re

if archive_ok('tad'):
    print('TAD archive verified -- skipped')
else:
    import kagglehub

    FARM = LOCAL / 'tad_farm'
    IMG = {'.jpg', '.jpeg', '.png', '.bmp'}
    t0 = time.time()
    root = Path(kagglehub.dataset_download(KAGGLE))
    print(f'Kaggle dataset at {root} ({time.time() - t0:.0f}s)')
    wanted, found = set(IDS['tad']), {}
    for dirpath, _, files in os.walk(root):
        if not any(Path(f).suffix.lower() in IMG for f in files):
            continue
        vid = re.sub(r'\.(mp4|avi|mov|mkv)$', '', Path(dirpath).name, flags=re.I)
        if vid in wanted:
            assert vid not in found, f'{vid} appears twice: {found[vid]} and {dirpath}'
            found[vid] = Path(dirpath)
    lost = sorted(wanted - set(found))
    assert not lost, f'{len(lost)} test videos have no frame folder in the Kaggle copy: {lost[:5]}'

    def key(p):
        digits = re.findall(r'\d+', p.stem)
        assert digits, f'no frame index in {p}'
        return int(digits[-1])

    shutil.rmtree(FARM, ignore_errors=True)
    for vid, src in sorted(found.items()):
        frames = sorted((p for p in src.iterdir() if p.suffix.lower() in IMG), key=key)
        idx = [key(p) for p in frames]
        assert len(set(idx)) == len(idx), f'{vid}: duplicate frame indices'
        assert math.ceil(len(frames) / STRIDE) == rows(TAD_NCC, vid), \
            f'{vid}: {len(frames)} frames -> ceil(N/8) != TAD_ncc rows {rows(TAD_NCC, vid)}'
        dst = FARM / vid
        dst.mkdir(parents=True)
        for i, p in enumerate(frames):
            (dst / f'{i:06d}{p.suffix.lower()}').symlink_to(p)
    print(f'farm: {len(found)} videos; ceil(N/8) == TAD_ncc rows on all')
    run([sys.executable, '-m', 'core.tools.zs_frames', 'pack', '--frames-dir', FARM, '--ids-file', ID_FILE['tad'],
         '--clip-dir', TAD_NCC, '--archive-dir', ARC['tad'], '--stride', STRIDE,
         '--report', REPORTS / 'zs_pack_tad.json'], log=LOCAL / 'pack_tad.log')
    run([sys.executable, '-m', 'core.tools.zs_frames', 'verify', '--dataset', constants.TAD_DATASET,
         '--ids-file', ID_FILE['tad'], '--clip-dir', TAD_NCC, '--archive-dir', ARC['tad'], '--stride', STRIDE,
         '--report', REPORTS / 'zs_verify_tad.json'])
    print(f'TAD archived in {(time.time() - t0) / 60:.1f} min')
assert archive_ok('tad')
print(json.loads((ARC['tad'] / constants.ZS_FRAMES_MANIFEST).read_text()))

## 5. DADA-2000 — row frames only, straight out of `DADA2000.zip` (≈ 15k PNG of ≈ 650k)

1. `7z l -slt` lists the archive once (central directory; cached in `REPORTS/`).
2. `zs_frames dada-plan`: per source, the image names of `DADA2000/{type}/{video:03d}/images/` sorted as
   `list_frame_images` sorts them (the CLIP cache's order), the count checked against the census (C10), then `[::8]`.
3. `7z x @list` extracts only those members (one pass over the volumes), the farm links each source's `images/` under
   its id, and `zs_frames pack --prestrided` archives them (gate: = the source's `DADA2000_orig` CLIP rows).

Pixel identity (same frame, same transform) is checked on the GPU side: the 1×1 cell must reproduce the CLIP row.

In [ ]:
if archive_ok('dada'):
    print('DADA archive verified -- skipped')
else:
    t0 = time.time()
    listing = REPORTS / 'dada2000_zip_listing.txt'
    if not listing.exists():
        local_listing = LOCAL / 'listing.txt'
        run(['7z', 'l', '-slt', ORIG / 'DADA2000.zip'], stdout_to=local_listing)
        shutil.copy2(local_listing, listing)
        print(f'listing: {local_listing.stat().st_size / 1e6:.0f} MB in {(time.time() - t0) / 60:.1f} min')
    plan_json, members = LOCAL / 'dada_plan.json', LOCAL / 'dada_members.txt'
    run([sys.executable, '-m', 'core.tools.zs_frames', 'dada-plan', '--listing', listing, '--census', T2_CENSUS,
         '--ids-file', ID_FILE['dada'], '--stride', STRIDE, '--out-list', members, '--out-plan', plan_json])
    plan = json.loads(plan_json.read_text())
    bad = {v: (len(p['members']), rows(T2_CLIP, v)) for v, p in plan.items() if len(p['members']) != rows(T2_CLIP, v)}
    assert not bad, f'planned row frames != DADA2000_orig CLIP rows on {len(bad)} sources: {list(bad.items())[:5]}'

    from core.tools.dota_cap_frames import archive_frames, archive_path

    todo = [v for v in IDS['dada'] if archive_frames(archive_path(ARC['dada'], v)) != rows(T2_CLIP, v)]
    print(f'{len(IDS["dada"]) - len(todo)}/{len(IDS["dada"])} sources already archived; extracting {len(todo)}')
    if todo:
        frames, farm = LOCAL / 'frames', LOCAL / 'farm'
        shutil.rmtree(frames, ignore_errors=True)
        shutil.rmtree(farm, ignore_errors=True)
        farm.mkdir(parents=True)
        todo_list = write_ids(LOCAL / 'todo_members.txt', [m for v in todo for m in plan[v]['members']])
        t1 = time.time()
        run(['7z', 'x', ORIG / 'DADA2000.zip', f'-o{frames}', '-y', '-bso0', '-bsp0', f'@{todo_list}'])
        print(f'7z extracted the row frames in {(time.time() - t1) / 60:.1f} min')
        for v in todo:                          # C26: each source's images dir under its unique id
            src = frames / plan[v]['folder']
            n = len([p for p in src.iterdir() if p.is_file()])
            assert n == len(plan[v]['members']), f'{v}: {n} files extracted, plan says {len(plan[v]["members"])}'
            (farm / v).symlink_to(src.resolve(), target_is_directory=True)
        run([sys.executable, '-m', 'core.tools.zs_frames', 'pack', '--frames-dir', farm, '--prestrided',
             '--ids-file', write_ids(LOCAL / 'dada_todo.txt', todo), '--clip-dir', T2_CLIP,
             '--archive-dir', ARC['dada'], '--stride', STRIDE, '--report', REPORTS / 'zs_pack_dada.json'],
            log=LOCAL / 'pack_dada.log')
        shutil.rmtree(frames, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.zs_frames', 'verify', '--dataset', T2NAME, '--ids-file', ID_FILE['dada'],
         '--clip-dir', T2_CLIP, '--archive-dir', ARC['dada'], '--stride', STRIDE,
         '--report', REPORTS / 'zs_verify_dada.json'])
    print(f'DADA archived in {(time.time() - t0) / 60:.1f} min')
assert archive_ok('dada')
print(json.loads((ARC['dada'] / constants.ZS_FRAMES_MANIFEST).read_text()))

In [ ]:
drive.flush_and_unmount()          # C10: every archive lands on Drive before the VM goes
print('Drive flushed -- now run colab/final/zs_tad_dada_r4.ipynb on a GPU runtime')